# Stance Scoring

## Imports

Load numeric utilities and dataframes.

In [1]:
import numpy as np
import pandas as pd

## Load Transformer Components

Bring in tokenizer/model classes and the pipeline helper.

In [2]:
from transformers import AutoTokenizer
from transformers import RobertaForSequenceClassification, RobertaTokenizer
from transformers import pipeline

/Users/dylanhuang/micromamba/envs/df_ae2/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Torch Setup

Import PyTorch for model inference.

In [3]:
import torch

## Progress Utility

Optionally use tqdm for progress display.

In [4]:
from tqdm.notebook import tqdm

## Load Emotion-Scored Tweets

Read the dataset containing sentiment and emotion features.

In [5]:
tweets = pd.read_parquet("../data/dataset/stock_tweets_sentiment_emotion_nomerge.parquet")

## Preview Tweets

Inspect a sample before stance scoring.

In [6]:
tweets

,ticker,text,created_at,user_id,date,sentiment,emotion_anger,emotion_disgust,emotion_fear,emotion_joy,...,emotion_joy_pct,emotion_neutral_pct,emotion_sadness_pct,emotion_surprize_pct,positive_emotion,negative_emotion,uncertainty_emotion,positive_emotion_pct,negative_emotion_pct,uncertainty_emotion_pct
0,AAPL,summary of yesterdays webcast featuring wynn g...,2013-12-31 23:10:08+00:00,1864753100,2013-12-31,4,0.677257,0.148463,0.143604,0.021908,...,0.341360,0.083108,0.214632,0.197516,0.021908,0.829146,0.144724,0.341360,1.168543,0.958575
1,AAPL,summary of yesterdays webcast featuring wynn g...,2014-01-01 01:18:36+00:00,1937591882,2014-01-01,4,0.677257,0.148463,0.143604,0.021908,...,0.341360,0.083108,0.214632,0.197516,0.021908,0.829146,0.144724,0.341360,1.168543,0.958575
2,AAPL,iphone users are more intelligent than samsung...,2014-01-01 01:52:31+00:00,23954327,2014-01-01,5,0.651749,0.290809,0.032565,0.017082,...,0.243422,0.074414,0.070304,0.430871,0.017082,0.944506,0.034343,0.243422,1.417922,0.611794
3,AAPL,summary of yesterdays webcast featuring wynn g...,2014-01-01 03:29:29+00:00,1933063572,2014-01-01,4,0.677257,0.148463,0.143604,0.021908,...,0.341360,0.083108,0.214632,0.197516,0.021908,0.829146,0.144724,0.341360,1.168543,0.958575
4,AAPL,summary of yesterdays webcast featuring wynn g...,2014-01-01 03:59:03+00:00,1938270918,2014-01-01,4,0.677257,0.148463,0.143604,0.021908,...,0.341360,0.083108,0.214632,0.197516,0.021908,0.829146,0.144724,0.341360,1.168543,0.958575
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
106333,XOM,t active morning movers at t nyse t exxon mobil …,2015-12-28 17:15:13+00:00,2342763212,2015-12-28,5,0.793153,0.089309,0.077659,0.018933,...,0.286050,0.474177,0.514482,0.468055,0.018933,0.889814,0.079564,0.286050,1.472964,0.938978
106334,XOM,divest from stopcommoncore optout because chil...,2015-12-28 19:39:46+00:00,4399710563,2015-12-28,1,0.763934,0.115369,0.071567,0.021422,...,0.333601,0.488936,0.602386,0.859171,0.021422,0.888479,0.077954,0.333601,1.595991,1.298266
106335,XOM,zsl stock forum zsl gold uslv zsl investing na...,2015-12-29 16:52:36+00:00,2181314366,2015-12-29,5,0.488881,0.242657,0.201586,0.045692,...,0.646072,0.500263,0.513626,0.298238,0.045692,0.738872,0.202946,0.646072,1.406148,1.224172
106336,XOM,nptn recent news updated tuesday december pm g...,2015-12-29 19:03:17+00:00,2197054086,2015-12-29,1,0.456476,0.355527,0.112046,0.067945,...,0.791015,0.059649,0.193402,0.154526,0.067945,0.815209,0.113054,0.791015,1.324451,0.799488


## Dataset Size

Check how many rows will be scored.

In [7]:
tweets.shape

(106338, 26)

## Load Stance Model

Initialize the StockTwits‑finetuned RoBERTa model and tokenizer.

In [8]:
tokenizer_loaded = RobertaTokenizer.from_pretrained('zhayunduo/roberta-base-stocktwits-finetuned')
model_loaded = RobertaForSequenceClassification.from_pretrained('zhayunduo/roberta-base-stocktwits-finetuned')


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 2373.81it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]              
RobertaForSequenceClassification LOAD REPORT from: zhayunduo/roberta-base-stocktwits-finetuned
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Select Device

Move the model to GPU if available.

In [9]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
bert_model = model_loaded.to(device)

## Confirm Device

Verify the model device.

In [10]:
torch.cuda.is_available()
print(next(bert_model.parameters()).device)

cpu


## Build Pipeline

Create a text-classification pipeline for stance inference.

In [11]:
nlp = pipeline("text-classification", model=bert_model, tokenizer=tokenizer_loaded)

## Stance Scoring Function

Return stance label and confidence for a given tweet.

In [12]:
def stance_score(text):
    result = nlp(text)
    label = result[0]['label']
    score = result[0]['score']


    
    return label, score

## Score All Tweets

Apply stance scoring to every tweet and expand into two columns.

In [13]:
tweets[['stance_label','stance_score']] = tweets['text'].apply(stance_score).apply(pd.Series)

## Preview Results

Inspect stance features.

In [14]:
tweets

,ticker,text,created_at,user_id,date,sentiment,emotion_anger,emotion_disgust,emotion_fear,emotion_joy,...,emotion_sadness_pct,emotion_surprize_pct,positive_emotion,negative_emotion,uncertainty_emotion,positive_emotion_pct,negative_emotion_pct,uncertainty_emotion_pct,stance_label,stance_score
0,AAPL,summary of yesterdays webcast featuring wynn g...,2013-12-31 23:10:08+00:00,1864753100,2013-12-31,4,0.677257,0.148463,0.143604,0.021908,...,0.214632,0.197516,0.021908,0.829146,0.144724,0.341360,1.168543,0.958575,Positive,0.742110
1,AAPL,summary of yesterdays webcast featuring wynn g...,2014-01-01 01:18:36+00:00,1937591882,2014-01-01,4,0.677257,0.148463,0.143604,0.021908,...,0.214632,0.197516,0.021908,0.829146,0.144724,0.341360,1.168543,0.958575,Positive,0.742110
2,AAPL,iphone users are more intelligent than samsung...,2014-01-01 01:52:31+00:00,23954327,2014-01-01,5,0.651749,0.290809,0.032565,0.017082,...,0.070304,0.430871,0.017082,0.944506,0.034343,0.243422,1.417922,0.611794,Positive,0.998410
3,AAPL,summary of yesterdays webcast featuring wynn g...,2014-01-01 03:29:29+00:00,1933063572,2014-01-01,4,0.677257,0.148463,0.143604,0.021908,...,0.214632,0.197516,0.021908,0.829146,0.144724,0.341360,1.168543,0.958575,Positive,0.742110
4,AAPL,summary of yesterdays webcast featuring wynn g...,2014-01-01 03:59:03+00:00,1938270918,2014-01-01,4,0.677257,0.148463,0.143604,0.021908,...,0.214632,0.197516,0.021908,0.829146,0.144724,0.341360,1.168543,0.958575,Positive,0.742110
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
106333,XOM,t active morning movers at t nyse t exxon mobil …,2015-12-28 17:15:13+00:00,2342763212,2015-12-28,5,0.793153,0.089309,0.077659,0.018933,...,0.514482,0.468055,0.018933,0.889814,0.079564,0.286050,1.472964,0.938978,Positive,0.803462
106334,XOM,divest from stopcommoncore optout because chil...,2015-12-28 19:39:46+00:00,4399710563,2015-12-28,1,0.763934,0.115369,0.071567,0.021422,...,0.602386,0.859171,0.021422,0.888479,0.077954,0.333601,1.595991,1.298266,Negative,0.949051
106335,XOM,zsl stock forum zsl gold uslv zsl investing na...,2015-12-29 16:52:36+00:00,2181314366,2015-12-29,5,0.488881,0.242657,0.201586,0.045692,...,0.513626,0.298238,0.045692,0.738872,0.202946,0.646072,1.406148,1.224172,Positive,0.997038
106336,XOM,nptn recent news updated tuesday december pm g...,2015-12-29 19:03:17+00:00,2197054086,2015-12-29,1,0.456476,0.355527,0.112046,0.067945,...,0.193402,0.154526,0.067945,0.815209,0.113054,0.791015,1.324451,0.799488,Positive,0.997328


## Test Example

Run a quick manual check on a synthetic message.

In [15]:
stance_score("buy buy buy!")

('Positive', 0.9862682819366455)

## Save Output

Write the stance‑scored dataset to parquet.

In [16]:
tweets.to_parquet('../data/dataset/stock_tweets_sentiment_emotion_stanceScore_nomerge.parquet',index=False)